# IDEA-003: Realized Loss Spike Entry

**Hypothesis:** Large spikes in realized losses = capitulation = buy.

**Logic:**
- Realized Loss = USD value of losses locked in by moving coins
- Spikes indicate panic selling at a loss
- Similar to SOPR < 1 but measures magnitude, not just direction
- Could catch the intensity of capitulation, not just presence

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Realized Loss Spike Exploration 🔍")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")

df = realized_loss.join(price, how='inner').join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner')
df = df.sort_index()

print(f"Data: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

---
## 1. Understand the Data

In [ ]:
# Basic stats
print("REALIZED LOSS STATISTICS")
print("="*50)
print(f"Min: ${df['realized_loss'].min():,.0f}")
print(f"Max: ${df['realized_loss'].max():,.0f}")
print(f"Mean: ${df['realized_loss'].mean():,.0f}")
print(f"Median: ${df['realized_loss'].median():,.0f}")
print(f"Current: ${df['realized_loss'].iloc[-1]:,.0f}")

print(f"\nPercentiles:")
for p in [50, 75, 90, 95, 99]:
    print(f"  {p}th: ${df['realized_loss'].quantile(p/100):,.0f}")

In [ ]:
# Create z-score for spike detection
# Use rolling stats to normalize for market size changes
df['rl_ma30'] = df['realized_loss'].rolling(30).mean()
df['rl_std30'] = df['realized_loss'].rolling(30).std()
df['rl_zscore'] = (df['realized_loss'] - df['rl_ma30']) / df['rl_std30']

# Also create percentile rank (rolling)
df['rl_percentile'] = df['realized_loss'].rolling(365).apply(lambda x: (x.iloc[-1] > x).mean() * 100, raw=False)

print("\nZ-SCORE STATISTICS (30-day rolling)")
print("="*50)
print(f"Min Z: {df['rl_zscore'].min():.2f}")
print(f"Max Z: {df['rl_zscore'].max():.2f}")
print(f"Current Z: {df['rl_zscore'].iloc[-1]:.2f}")

In [ ]:
# Visualize
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.4, 0.3, 0.3],
                    subplot_titles=['BTC Price', 'Realized Loss (USD)', 'Realized Loss Z-Score'])

fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['realized_loss'], name='Realized Loss',
                         line=dict(color='red')), row=2, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['rl_zscore'], name='Z-Score',
                         line=dict(color='purple')), row=3, col=1)
fig.add_hline(y=2, line_dash='dash', line_color='orange', row=3, col=1)
fig.add_hline(y=3, line_dash='dash', line_color='red', row=3, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_yaxes(type='log', row=2, col=1)
fig.update_layout(height=700, title_text='Realized Loss - Spikes = Capitulation?')
fig.show()

In [ ]:
# Find major spike events
spikes = df[df['rl_zscore'] > 3].copy()
print(f"\nMAJOR REALIZED LOSS SPIKES (Z > 3)")
print("="*60)
print(f"Total spike days: {len(spikes)}")

if len(spikes) > 0:
    spikes['gap'] = (spikes.index.to_series().diff() > pd.Timedelta(days=7)).cumsum()
    spike_periods = spikes.groupby('gap').agg({
        'rl_zscore': 'max',
        'realized_loss': 'max',
        'price': ['first', 'min']
    })
    spike_periods.columns = ['max_z', 'max_loss', 'price_at_spike', 'min_price']
    
    period_dates = spikes.groupby('gap').apply(lambda x: (x.index.min(), x.index.max()))
    
    print(f"\nDistinct spike periods: {len(spike_periods)}")
    print("\n" + "-"*80)
    for i, (idx, row) in enumerate(spike_periods.iterrows()):
        start, end = period_dates.iloc[i]
        print(f"{start.date()} to {end.date()}: Z={row['max_z']:.1f}, Loss=${row['max_loss']/1e9:.1f}B, Price=${row['price_at_spike']:,.0f}")

---
## 2. Compare to SOPR Signal

In [ ]:
# Correlation
print("CORRELATION ANALYSIS")
print("="*50)
print(f"Realized Loss vs SOPR: {df['realized_loss'].corr(df['sopr']):.3f}")
print(f"RL Z-Score vs SOPR: {df['rl_zscore'].corr(df['sopr']):.3f}")

# When SOPR signals fire, what's the realized loss?
sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
print(f"\nWhen SOPR double cap fires:")
print(f"  Avg RL Z-Score: {df.loc[sopr_signal, 'rl_zscore'].mean():.2f}")
print(f"  Max RL Z-Score: {df.loc[sopr_signal, 'rl_zscore'].max():.2f}")

# When RL spikes, does SOPR also signal?
rl_spike = df['rl_zscore'] > 2
if rl_spike.sum() > 0:
    sopr_overlap = (rl_spike & sopr_signal).sum() / rl_spike.sum() * 100
    print(f"\nWhen RL Z > 2:")
    print(f"  SOPR also signaling: {sopr_overlap:.1f}% of the time")

---
## 3. Test Realized Loss as Entry Signal

In [ ]:
# Backtest framework
df_test = df[df.index >= '2018-12-15'].copy()
df_test = df_test.dropna(subset=['rl_zscore'])  # Need rolling stats
close = df_test['price']

def create_entry_signal(df, z_threshold):
    """Entry when RL z-score spikes above threshold (first day)."""
    above = df['rl_zscore'] > z_threshold
    entries = above & ~above.shift(1).fillna(False)
    return entries

In [ ]:
def backtest_mvrv_trailing(
    df, entries,
    mvrv_trigger=2.25,
    trailing_pct=0.20,
    stop_loss=0.20,
    max_hold_days=365
):
    """Same exit strategy as our best SOPR strategy."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            if not trailing_active and current_mvrv >= mvrv_trigger:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason,
            'entry_rl_z': df.loc[entry_date, 'rl_zscore']
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Test different z-score thresholds
thresholds = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]

print("REALIZED LOSS Z-SCORE THRESHOLD COMPARISON (In-Sample)")
print("="*100)
print(f"{'Threshold':<12} {'Signals':>10} {'Trades':>10} {'Return':>12} {'Win Rate':>10} {'Avg Days':>10}")
print("-"*100)

rl_results = []

for thresh in thresholds:
    entries = create_entry_signal(df_test, thresh)
    n_signals = entries.sum()
    
    if n_signals == 0:
        print(f"Z > {thresh:<7} {n_signals:>10} {'-':>10} {'-':>12} {'-':>10} {'-':>10}")
        continue
    
    trades = backtest_mvrv_trailing(df_test, entries)
    
    total_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
    win_rate = (trades['pnl_pct'] > 0).mean() if len(trades) > 0 else 0
    avg_days = trades['days_held'].mean() if len(trades) > 0 else 0
    
    print(f"Z > {thresh:<7} {n_signals:>10} {len(trades):>10} {total_return*100:>11.0f}% "
          f"{win_rate*100:>9.0f}% {avg_days:>10.0f}")
    
    rl_results.append({
        'threshold': thresh,
        'signals': n_signals,
        'trades': len(trades),
        'total_return': total_return,
        'win_rate': win_rate,
        'trades_df': trades
    })

In [ ]:
# Show trades for interesting threshold
if len(rl_results) > 0:
    best_is = max(rl_results, key=lambda x: x['total_return'])
    print(f"\nBest in-sample: Z > {best_is['threshold']}")
    print(f"Total return: {best_is['total_return']*100:.0f}%")
    print(f"\nTrades:")
    display_trades = best_is['trades_df'].copy()
    display_trades['entry_date'] = pd.to_datetime(display_trades['entry_date']).dt.strftime('%Y-%m-%d')
    display_trades['exit_date'] = pd.to_datetime(display_trades['exit_date']).dt.strftime('%Y-%m-%d')
    display_trades['pnl_pct'] = (display_trades['pnl_pct'] * 100).round(1)
    display_trades['entry_rl_z'] = display_trades['entry_rl_z'].round(2)
    print(display_trades[['entry_date', 'exit_date', 'entry_price', 'exit_price', 'pnl_pct', 'exit_reason', 'entry_rl_z']].to_string(index=False))

---
## 4. Walk-Forward Validation

In [ ]:
def walk_forward(df, threshold, mvrv_trigger=2.25, trailing_pct=0.20):
    """Walk-forward validation."""
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        entries = create_entry_signal(test_df, threshold)
        trades = backtest_mvrv_trailing(test_df, entries, mvrv_trigger, trailing_pct)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return,
            'n_trades': len(trades)
        })
    
    wf_df = pd.DataFrame(results)
    return wf_df['beat_hold'].mean(), (wf_df['strat_return'] - wf_df['hold_return']).mean(), wf_df['n_trades'].sum()

In [ ]:
# Walk-forward for each threshold
print("\nWALK-FORWARD VALIDATION")
print("="*80)
print(f"{'Threshold':<12} {'Beat Rate':>15} {'Avg Excess':>15} {'Total Trades':>15}")
print("-"*80)

wf_results = []

for thresh in thresholds:
    beat_rate, avg_excess, total_trades = walk_forward(df_test, thresh)
    
    print(f"Z > {thresh:<7} {beat_rate*100:>14.0f}% {avg_excess*100:>+14.1f}% {total_trades:>15}")
    
    wf_results.append({
        'threshold': thresh,
        'beat_rate': beat_rate,
        'avg_excess': avg_excess,
        'total_trades': total_trades
    })

wf_df = pd.DataFrame(wf_results)

In [ ]:
# Visualize
valid_wf = wf_df[wf_df['total_trades'] > 0]

if len(valid_wf) > 0:
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=[f"Z > {t}" for t in valid_wf['threshold']],
        y=valid_wf['beat_rate'] * 100,
        marker_color=['green' if x > 0.62 else 'orange' if x > 0.54 else 'gray' for x in valid_wf['beat_rate']],
        text=[f"{x:.0f}%" for x in valid_wf['beat_rate']*100],
        textposition='outside'
    ))

    fig.add_hline(y=54, line_dash='dash', line_color='orange', annotation_text='SOPR baseline 54%')
    fig.add_hline(y=62, line_dash='dash', line_color='green', annotation_text='SOPR+MVRV 62%')

    fig.update_layout(
        title='Walk-Forward Beat Rate by Realized Loss Z-Score Threshold',
        yaxis_title='Beat Rate %',
        height=500
    )
    fig.show()

---
## 5. Combine Realized Loss with SOPR?

In [ ]:
# SOPR + high realized loss
def combined_entry(df, rl_z_threshold):
    """Entry when SOPR double cap AND realized loss spike."""
    sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
    rl_signal = df['rl_zscore'] > rl_z_threshold
    combined = sopr_signal & rl_signal
    entries = combined & ~combined.shift(1).fillna(False)
    return entries

print("COMBINED SIGNAL: SOPR + REALIZED LOSS")
print("="*80)
print(f"{'RL Z Threshold':<15} {'Signals':>10} {'Beat Rate':>15} {'Avg Excess':>15}")
print("-"*80)

combined_results = []

for rl_z in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    # Walk-forward
    results = []
    close = df_test['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df_test)
    n_folds = (total_days - train_days) // step_days
    total_signals = 0
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df_test.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        entries = combined_entry(test_df, rl_z)
        total_signals += entries.sum()
        trades = backtest_mvrv_trailing(test_df, entries)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    wf_result = pd.DataFrame(results)
    beat_rate = wf_result['beat_hold'].mean()
    avg_excess = (wf_result['strat_return'] - wf_result['hold_return']).mean()
    
    print(f"SOPR + RL Z>{rl_z:<4} {total_signals:>10} {beat_rate*100:>14.0f}% {avg_excess*100:>+14.1f}%")
    
    combined_results.append({
        'rl_z_threshold': rl_z,
        'signals': total_signals,
        'beat_rate': beat_rate,
        'avg_excess': avg_excess
    })

---
## 6. Summary

In [ ]:
print("\n" + "="*80)
print("REALIZED LOSS SPIKE ANALYSIS SUMMARY")
print("="*80)

# Best standalone
valid_wf = wf_df[wf_df['total_trades'] > 0]
if len(valid_wf) > 0:
    best_rl = valid_wf.loc[valid_wf['beat_rate'].idxmax()]
    print(f"\n📊 STANDALONE REALIZED LOSS ENTRY")
    print(f"   Best threshold: Z > {best_rl['threshold']}")
    print(f"   Beat rate: {best_rl['beat_rate']*100:.0f}%")
    print(f"   Avg excess: {best_rl['avg_excess']*100:+.1f}%")
else:
    print(f"\n📊 STANDALONE REALIZED LOSS ENTRY")
    print(f"   ⚠️ Not enough signals")
    best_rl = None

# Best combined
combined_df = pd.DataFrame(combined_results)
valid_combined = combined_df[combined_df['signals'] > 0]
if len(valid_combined) > 0:
    best_combined = valid_combined.loc[valid_combined['beat_rate'].idxmax()]
    print(f"\n📊 COMBINED SIGNAL (SOPR + RL)")
    print(f"   Best threshold: SOPR + RL Z > {best_combined['rl_z_threshold']}")
    print(f"   Beat rate: {best_combined['beat_rate']*100:.0f}%")
    print(f"   Avg excess: {best_combined['avg_excess']*100:+.1f}%")
else:
    print(f"\n📊 COMBINED SIGNAL (SOPR + RL)")
    print(f"   ⚠️ Not enough signals")
    best_combined = None

# Comparison
print(f"\n📊 COMPARISON")
print(f"   SOPR + MVRV exit:  62% beat rate (current best)")
if best_rl is not None:
    print(f"   RL spike entry:    {best_rl['beat_rate']*100:.0f}% beat rate")
if best_combined is not None:
    print(f"   SOPR + RL filter:  {best_combined['beat_rate']*100:.0f}% beat rate")

# Verdict
print(f"\n🎯 VERDICT:")
improvement = False
if best_rl is not None and best_rl['beat_rate'] > 0.62:
    print(f"   ✅ Realized Loss entry BEATS SOPR!")
    improvement = True
if best_combined is not None and best_combined['beat_rate'] > 0.62:
    print(f"   ✅ Combined signal BEATS baseline!")
    improvement = True
if not improvement:
    print(f"   ⚠️ Realized Loss doesn't improve on current best.")

print("\n" + "="*80)

In [ ]:
# Save results
import json

def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_native(v) for v in obj]
    elif hasattr(obj, 'item'):
        return obj.item()
    return obj

results = {
    'signal': 'realized_loss_spike',
    'standalone_results': to_native(wf_df.to_dict('records')),
    'combined_results': to_native(combined_results),
    'best_standalone': to_native(dict(best_rl)) if best_rl is not None else None,
    'best_combined': to_native(dict(best_combined)) if best_combined is not None else None,
    'baseline_sopr': {'beat_rate': 0.62}
}

with open('../data/realized_loss_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to ../data/realized_loss_results.json")